# SATELLITE-X v0.8.0 — Google Colab Independent Verification

ఈ notebook software tests, live external-source tests, independent raster/formula oracles, security checks, privacy suppression, scene-aligned weather, authenticated farmer flow—all verify చేస్తుంది.

**Important:**
- First upload `SATELLITE-X_v0.8.0_COLAB_READY.zip`.
- Live cells need Colab internet access.
- Government API, real SMS, field hardware, verified yield labels, and machinery trials are not fabricated. Their production paths remain fail-closed without real credentials/evidence.
- Run cells from top to bottom. A failed assertion means verification is not complete.


In [ ]:
# 1. Upload and extract the SATELLITE-X v0.8.0 release ZIP
from google.colab import files
from pathlib import Path
import os, zipfile, shutil, json, subprocess, sys, time

uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_files, "Upload SATELLITE-X_v0.8.0_COLAB_READY.zip"
zip_name = zip_files[0]
extract_root = Path('/content/satellite_x_release')
if extract_root.exists(): shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as archive:
    bad = archive.testzip()
    assert bad is None, f"Corrupt ZIP member: {bad}"
    archive.extractall(extract_root)
projects = list(extract_root.glob('*/pyproject.toml')) + list(extract_root.glob('pyproject.toml'))
assert projects, "pyproject.toml not found after extraction"
project = projects[0].parent
os.chdir(project)
print('Project:', project)
print('Files:', sum(1 for p in project.rglob('*') if p.is_file()))


In [ ]:
# 2. Install exact project requirements and editable package
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
os.environ['PYTHONPATH'] = str(project / 'src')
if shutil.which('node') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'nodejs'], check=True)
import satellite_x
print('SATELLITE-X version:', satellite_x.__version__)
assert satellite_x.__version__ == '0.8.0'
print('Node:', subprocess.check_output(['node', '--version'], text=True).strip())


In [ ]:
# Helper: execute a command, stream output, and fail immediately on any non-zero exit
def run(cmd, timeout=900):
    print('\n$', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout, env={**os.environ, 'PYTHONPATH': str(project/'src')})
    print(result.stdout)
    assert result.returncode == 0, f"Command failed with exit {result.returncode}: {' '.join(cmd)}"
    return result.stdout


## Gate A — Deterministic software tests
Expected: **95 passed**, live tests excluded by project configuration.

In [ ]:
deterministic_output = run([sys.executable, '-m', 'pytest', '-q'], timeout=600)
assert 'failed' not in deterministic_output.lower()


## Gate B — Real live integration tests
Calls real public APIs/COGs. Expected: **8 passed**. Temporary upstream outages should be treated as an external availability failure—not silently converted into fake success.

In [ ]:
live_output = run([sys.executable, '-m', 'pytest', '-m', 'live', '-o', 'addopts=', '-q', '-vv'], timeout=1200)
assert '8 passed' in live_output


## Gate C — JSON schemas and representative output validation

In [ ]:
from jsonschema import Draft202012Validator, validate
run([sys.executable, '-m', 'satellite_x', 'export-schemas', '--directory', 'schemas'], timeout=120)
schema_files = sorted(Path('schemas').glob('*.json'))
assert len(schema_files) == 32, f"Expected 32 schemas, got {len(schema_files)}"
for path in schema_files:
    Draft202012Validator.check_schema(json.loads(path.read_text()))
pairs = {
    'analytics_result.schema.json': 'outputs/analytics_crop_field_result.json',
    'diagnosis_result.schema.json': 'outputs/diagnosis_crop_field_result.json',
    'timeseries_result.schema.json': 'outputs/timeseries_live_result.json',
    'sar_fallback_result.schema.json': 'outputs/sar_fallback_crop_field_result.json',
    'yield_model_candidate.schema.json': 'outputs/yield_candidate_validation_only.json',
}
for schema_name, output_name in pairs.items():
    validate(json.loads(Path(output_name).read_text()), json.loads((Path('schemas')/schema_name).read_text()))
print('PASS: 32 schemas structurally valid; representative outputs validate')


## Gate D — Independent geospatial/formula oracles

In [ ]:
run([sys.executable, 'audit/preprocessing_reality.py', '--input', 'outputs/preprocessing_crop_field_result.json', '--report', 'outputs/preprocessing_reality.json'], timeout=600)
run([sys.executable, 'audit/analytics_diagnosis_reality.py', '--preprocessing', 'outputs/preprocessing_crop_field_result.json', '--set1', 'outputs/set1_crop_field_scene_aligned.json', '--analytics', 'outputs/analytics_crop_field_result.json', '--diagnosis', 'outputs/diagnosis_crop_field_result.json', '--report', 'outputs/analytics_diagnosis_reality.json'], timeout=600)
run([sys.executable, 'audit/timeseries_reality.py'], timeout=120)
run([sys.executable, 'audit/sar_reality.py'], timeout=600)
assert json.loads(Path('outputs/preprocessing_reality.json').read_text())['pass'] is True
assert json.loads(Path('outputs/analytics_diagnosis_reality.json').read_text())['pass'] is True
assert json.loads(Path('outputs/timeseries_reality.json').read_text())['passed'] is True
assert json.loads(Path('outputs/sar_reality.json').read_text())['passed'] is True
print('PASS: Set2 13/13, Set3/4 12/12, time-series 16/16, SAR 5/5')


## Gate D2 — Orbit, Doppler, ITU-R and 120-station scheduled-contact validation

Verifies TLE checksums, SGP4 pass geometry, dynamic Doppler, atmospheric contributions, calibrated-input link math, fail-closed historical TLE policy and DES conservation. These are not called live beacon telemetry or ESA operational traffic.


In [ ]:
run([sys.executable, 'audit/orbit_communications_reality.py'], timeout=300)
orbit_comms = json.loads(Path('outputs/orbit_communications_reality.json').read_text())
assert orbit_comms['passed'] is True
assert orbit_comms['check_count'] == 16
atmos = json.loads(Path('outputs/atmospheric_loss_guntur_xband_validation.json').read_text())
traffic = json.loads(Path('outputs/traffic_120_station_validation_only.json').read_text())
provenance = json.loads(Path('outputs/scene_orbit_provenance.json').read_text())
assert atmos['live_beacon_calibrated'] is False
assert traffic['scenario_purpose'] == 'deterministic_validation_fixture'
assert provenance['status'] == 'historical_tle_required'
print('PASS: 16/16 independent orbit/communications checks')
print('Atmospheric modeled total dB:', atmos['modeled_total_db'])
print('Traffic completed/dropped:', traffic['completed_requests'], traffic['dropped_requests'])


## Gate E — Material scene/weather alignment check
This prevents combining old optical pixels with unrelated current weather.

In [ ]:
analytics = json.loads(Path('outputs/analytics_crop_field_result.json').read_text())
water = analytics['water_balance']
print(json.dumps({'scene_date': analytics['scene_date'], 'weather_start': water['reference_start_date'], 'weather_end': water['reference_end_date'], 'scene_alignment_days': water['scene_alignment_days'], 'rain_15d_mm': water['rain_15d_mm'], 'et0_15d_mm': water['et0_15d_mm'], 'balance_mm': water['water_balance_15d_mm'], 'deficit_flag': water['deficit_flag']}, indent=2))
assert water['reference_end_date'] == analytics['scene_date']
assert water['scene_alignment_days'] == 0
assert water['water_balance_15d_mm'] == -12.67
assert water['deficit_flag'] is False
print('PASS: optical and weather are scene-aligned')


## Gate F — Privacy suppression check
One-field aggregates must release no count, area, crop or verdict metrics.

In [ ]:
village = json.loads(Path('outputs/village_summary_demo.json').read_text())
print(json.dumps(village, indent=2))
assert village['privacy_status'] == 'suppressed_small_group'
assert village['minimum_group_size'] == 5
assert village['field_count'] is None
assert village['total_agri_acres'] is None
assert village['crop_acres'] == {} and village['verdict_acres'] == {}
print('PASS: k>=5 small-group suppression')


## Gate G — Mobile/API/security verification
Uses deterministic test SMS transport only; no fake production SMS claim. Verifies encrypted photo requirement, OTP/session flow, rate/body controls, Ed25519 receipts, JavaScript verification, XSS-safe DOM and service-worker API bypass.

In [ ]:
run([sys.executable, 'audit/mobile_sync_reality.py'], timeout=120)
run([sys.executable, 'audit/api_mobile_flow.py'], timeout=180)
mobile = json.loads(Path('outputs/mobile_sync_reality.json').read_text())
api = json.loads(Path('outputs/api_mobile_flow.json').read_text())
assert mobile['passed'] and api['passed']
print('Mobile checks:', len(mobile['checks']), '/', len(mobile['checks']))
print('API checks:', len(api['checks']), '/', len(api['checks']))


## Gate H — Streamlit end-to-end flows
The first is explicit demo mode. The second proves authenticated login, boundary confirmation hash link, own-field enforcement, scene-aligned weather and Set 2–4 execution.

In [ ]:
run([sys.executable, 'audit/streamlit_flow.py'], timeout=1200)
run([sys.executable, 'audit/streamlit_authenticated_flow.py'], timeout=1200)
demo = json.loads(Path('outputs/streamlit_flow.json').read_text())
auth = json.loads(Path('outputs/streamlit_authenticated_flow.json').read_text())
assert demo['passed'] and auth['passed']
assert auth['analyzed_field'] in auth['owned_field_ids']
assert auth['weather_reference_end'] == auth['scene_date']
print('PASS: demo and authenticated Streamlit flows')


## Gate I — Known-gap register
This check ensures verification does not hide unresolved external/scientific work.

In [ ]:
gap = Path('POWER_ENGINE_GAP_REGISTER_2026-08-18.md').read_text()
assert 'P0 — Critical loopholes' in gap
assert 'P1 — High-priority science' in gap
assert 'Direct delay sources' in gap
print('Gap register present:', len(gap.splitlines()), 'lines')
print('IMPORTANT: external credentials, real labels, field trials and device acceptance remain external inputs.')


## Optional — test your own field input
Set `RUN_CUSTOM_FIELD=True` only after replacing every value. A real boundary is required. This cell runs Set 1 only; it does not claim legal ownership or diagnosis.

In [ ]:
RUN_CUSTOM_FIELD = False
my_field = {
    'field_id': 'REPLACE_FIELD_ID',
    'latitude': 16.0,
    'longitude': 80.0,
    'crop_type': 'chilli',
    'sowing_date': '2026-06-15',
    'analysis_date': '2026-08-18',
    'scan_range_days': 30,
    'acres': 2.0,
    'boundary_geojson': {
        'type': 'Polygon',
        'coordinates': [[[80.0,16.0],[80.001,16.0],[80.001,16.001],[80.0,16.001],[80.0,16.0]]],
    },
}
if RUN_CUSTOM_FIELD:
    Path('my_field.json').write_text(json.dumps(my_field, indent=2))
    run([sys.executable, '-m', 'satellite_x', 'acquire', '--input', 'my_field.json', '--output', 'outputs/my_field_set1.json'], timeout=600)
else:
    print('Skipped. Replace all values and set RUN_CUSTOM_FIELD=True when ready.')


## Final machine-readable Colab summary and download

In [ ]:
summary = {
    'version': satellite_x.__version__,
    'deterministic_tests': 'PASS',
    'live_tests': 'PASS',
    'schemas_32': 'PASS',
    'set2_oracle': json.loads(Path('outputs/preprocessing_reality.json').read_text())['pass'],
    'set3_set4_oracle': json.loads(Path('outputs/analytics_diagnosis_reality.json').read_text())['pass'],
    'timeseries_oracle': json.loads(Path('outputs/timeseries_reality.json').read_text())['passed'],
    'sar_oracle': json.loads(Path('outputs/sar_reality.json').read_text())['passed'],
    'orbit_communications_oracle': json.loads(Path('outputs/orbit_communications_reality.json').read_text())['passed'],
    'mobile_security': json.loads(Path('outputs/mobile_sync_reality.json').read_text())['passed'],
    'api_security': json.loads(Path('outputs/api_mobile_flow.json').read_text())['passed'],
    'streamlit_demo': json.loads(Path('outputs/streamlit_flow.json').read_text())['passed'],
    'streamlit_authenticated': json.loads(Path('outputs/streamlit_authenticated_flow.json').read_text())['passed'],
    'weather_scene_aligned': water['reference_end_date'] == analytics['scene_date'],
    'privacy_small_group_suppressed': village['privacy_status'] == 'suppressed_small_group',
    'external_activation_complete': False,
    'external_activation_note': 'Requires real government/SMS credentials, verified labels, hardware and field trials; never fabricated.',
}
Path('outputs/colab_verification_summary.json').write_text(json.dumps(summary, indent=2)+'\n')
print(json.dumps(summary, indent=2))
assert all(value is True or value == 'PASS' for key,value in summary.items() if key not in {'version','external_activation_complete','external_activation_note'})
files.download('outputs/colab_verification_summary.json')
